# Synthetic stage: priors, prior predictive check, and parameter recovery

**Nothing in this notebook has been run.** Every output cell is empty by design, and no figure is written to disk. Run the cells one at a time, in order, and stop when something looks odd.

The notebook states what each cell measures. It does not tell you what the numbers mean.

## What is here

| Section | Contents |
|---|---|
| 0 | Config: paths, seed, draw counts, thresholds, plotting |
| 1 | Prior table with provenance |
| 2 | Prior predictive check on the real FI-Hyy drivers |
| 3 | Branches: what each prior-check outcome implies for the next step |
| 4 | True parameter set and synthetic dataset |
| 5 | Posterior: R-hat and effective sample size |
| 6 | Recovery: posterior against the true value |
| 7 | Prior against posterior, per parameter |
| 8 | Pairwise correlation |
| 9 | Posterior predictive check and residuals |
| 10 | Open questions, for you to fill in |
| 11 | Self-critique of this notebook's code |

## Terms used before their own section

- **Prior**: the distribution of a parameter before any data is seen.
- **Posterior**: that distribution after conditioning on the data.
- **Draw**: one sampled parameter set, all 24 fields at once.
- **Identifiability**: whether the data can pin a parameter down at all. A parameter is unidentifiable when its posterior stays at its prior no matter how much data is added.

## What runs today, and what does not

Sections 0 to 3 read files that exist in this repository: `config/default.yaml` and the processed calibration block under `data/processed/`.

Sections 4 to 9 read the synthetic twin: a true parameter set, a synthetic NEE series, and a posterior sampled against it. **`scripts/03_synthetic_twin.py` is not built yet** -- it raises `SystemExit` on import -- so those files do not exist and those cells will stop with a clear message until the script is written. The filenames in the config cell are this notebook's assumption about what that script will produce; change them to match whatever it actually writes.

---

# 0. Config

Everything you might want to change lives in the next four cells and nowhere else. No path, threshold or count is retyped further down.

One rule inherited from the codebase: **no prior bound is typed in this notebook.** Bounds come out of `dalec.priors.prior_sources`, `dalec.parameters.canopy_bounds` and `dalec.diagnostics.reparameterised_bounds`, which read them from their recorded sources. What this notebook does type is the provenance metadata -- symbol, unit, citation, category -- which lives nowhere else yet.

In [ ]:
# Standard library, third party, and the repository root.
#
# The root is found by walking up from the working directory until config/default.yaml
# appears, so the notebook runs the same whether Jupyter was started in notebooks/ or
# at the repository root.
import json
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'config' / 'default.yaml').is_file():
            return candidate
    raise FileNotFoundError(
        'could not find the repository root: no config/default.yaml in this '
        'directory or any parent of ' + str(here)
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print('repository root:', REPO_ROOT)
print('python:', sys.version.split()[0])

In [ ]:
# Project imports. Every one of these names is public API of src/dalec.
from dalec.acm import acm_from_config, leaf_area_index
from dalec.config import load_config, require_year_block, resolve_path
from dalec.data_io import SiteData
from dalec.diagnostics import (
    classify_prior_draw,
    reparameterised_bounds,
    sample_reparameterised_parameters,
)
from dalec.model_numpy import POOL_NAMES, run_dalec2
from dalec.parameters import (
    ALLOCATION_PARAMETERS,
    ALLOCATION_WEIGHT_ORDER,
    PARAMETER_NAMES,
    PARAMETER_REGISTRY,
    DalecParameters,
    allocation_concentration,
    canopy_bounds,
    site_lai,
)
from dalec.plotting import OKABE_ITO, apply_style
from dalec.priors import prior_sources

print('dalec imports ok')
print('model parameters:', len(PARAMETER_NAMES))
print('carbon pools:', POOL_NAMES)

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 1 of 3 -- paths
# ---------------------------------------------------------------------------

CONFIG_PATH = REPO_ROOT / 'config' / 'default.yaml'

# LAI convention. Both stay live in this project and neither is adopted; it
# changes lma and nothing else. 'hemisurface' or 'projected'.
CONVENTION = 'hemisurface'

# --- synthetic twin artefacts ----------------------------------------------
# TODO: scripts/03_synthetic_twin.py is not built yet, so none of these files
# exists and none of these names is confirmed. They are this notebook's
# assumption about what Phase 7 will write. Point them at the real filenames
# once the script exists.
SYNTHETIC_DIR = REPO_ROOT / 'results' / 'synthetic'

# The known parameter set the synthetic data was generated from.
# Assumed format: a flat JSON object, one key per DalecParameters field, as
# produced by DalecParameters.to_dict(). A nested {'parameters': {...}} object
# is also accepted by the loader in section 4.
TRUE_PARAMS_PATH = SYNTHETIC_DIR / 'synthetic_truth.json'

# The synthetic dataset: a SiteData NetCDF whose nee_obs is model output plus
# noise rather than the measured series. Assumed to be written with
# SiteData.save, so SiteData.load reads it back.
SYNTHETIC_BLOCK_PATH = SYNTHETIC_DIR / 'synthetic_block.nc'

# The posterior sampled against that synthetic dataset, written through ArviZ.
SYNTHETIC_TRACE_PATH = SYNTHETIC_DIR / 'synthetic_trace.nc'

# Optional cache of daily posterior predictive NEE, one row per posterior draw.
# Section 9 recomputes it if this file is absent.
SYNTHETIC_POSTPRED_PATH = SYNTHETIC_DIR / 'synthetic_posterior_predictive.npz'

# To exercise sections 5, 7 and 8 against the existing real-data trace before
# the synthetic twin exists, set this to
#     REPO_ROOT / 'results' / 'calibration_hemisurface.nc'
# Sections 4, 6 and 9 still need the true parameters and the synthetic NEE and
# will not run. That trace is recorded in the repository as NOT converged
# (r-hat 1.50, two modes), so nothing read from it is a posterior statement.
TRACE_PATH_OVERRIDE = None

TRACE_PATH = TRACE_PATH_OVERRIDE or SYNTHETIC_TRACE_PATH

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 2 of 3 -- run settings and plausibility thresholds
# ---------------------------------------------------------------------------

# Prior predictive draws. Start at 200; raise once the machinery is checked.
# NOTE: the draws are NOT nested. Each parameter is drawn as one vector of
# length N_PRIOR_DRAWS, so changing this number changes every draw, not just
# how many there are. Results at 200 and at 1000 are different samples of the
# same prior, not a subset and a superset.
N_PRIOR_DRAWS = 200

# Restrict the prior predictive run to these calibration years, inclusive, to
# keep the first pass quick. Set to None to use the whole calibration block.
# The forward model is a per-day Python loop, so cost is linear in
# draws x days.
PRIOR_PRED_YEARS = (1997, 1999)

# The single year drawn in the fan plot. Must lie inside PRIOR_PRED_YEARS.
FAN_YEAR = 1998

# Percentiles of the simulated NEE fan, and of the observed reference band.
FAN_PERCENTILES = (5.0, 25.0, 50.0, 75.0, 95.0)
OBS_BAND_PERCENTILES = (0.0, 100.0)

# Store the full daily GPP series as well as NEE. Costs
# N_PRIOR_DRAWS x n_days x 8 bytes.
STORE_FULL_GPP = True

# Progress line every this many draws during the forward run.
PROGRESS_EVERY = 25

# --- physical plausibility thresholds --------------------------------------
# Each of these is a stated assumption, not a measurement. Change them here.

# GPP floor. Gross primary production is an uptake and cannot be negative.
GPP_FLOOR_G_C_M2_D = 0.0

# LAI ceiling, on the basis set by CONVENTION. Filled in from the site's
# measured LAI in the next cell; SITE_LAI_MULTIPLE says how far above the
# measured seasonal maximum a draw is allowed to go before it is counted as a
# failure.
# TODO: 2.0 is a round number chosen here, not a published tolerance. Decide
# what multiple you are willing to defend, or replace the whole rule with the
# fixed band dalec.diagnostics uses for the GPP magnitude gate, DEFAULT_LAI_BAND
# = (1.0, 8.0).
SITE_LAI_MULTIPLE = 2.0

# Plausible annual NEE window, g C m-2 yr-1, negative being net uptake.
# TODO: assumption. The repository records the measured annual NEE at FI-Hyy in
# two places and they do not agree: -215.8 (scripts/22_calibration_figures.py,
# attributed to Ilvesniemi et al. 2009 Fig. 6) and about -206
# (src/dalec/parameters.py, in the litter-input comment). The window below is
# deliberately much wider than either, so that it screens out impossible
# ecosystems rather than merely atypical years. Set it to what you will defend.
ANNUAL_NEE_RANGE_G_C_M2_YR = (-600.0, 300.0)

# A calendar year needs at least this many days in the block to be summed.
MIN_DAYS_FOR_ANNUAL = 365

# --- section 5 to 9 settings ------------------------------------------------
# Posterior draws used for the posterior predictive recomputation in section 9.
# Thinned evenly across every chain.
N_POSTPRED_DRAWS = 100

# How many parameter pairs to list in section 8.
N_TOP_PAIRS = 15

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG 3 of 3 -- load the project config, resolve the derived settings
# ---------------------------------------------------------------------------
apply_style()          # the project matplotlib style, from src/dalec/plotting.py
warnings.filterwarnings('ignore', category=RuntimeWarning)

CONFIG = load_config(CONFIG_PATH)

SEED = int(CONFIG['seed'])
CALIBRATION_YEARS = require_year_block(CONFIG, 'calibration')
HDI_PROB = float(CONFIG['diagnostics']['hdi_prob'])
RHAT_THRESHOLD = float(CONFIG['diagnostics']['rhat_threshold'])
SITE_CODE = str(CONFIG['site']['code'])
SITE_SLUG = SITE_CODE.lower().replace('-', '_')

PROCESSED_DIR = resolve_path(CONFIG['paths']['processed_dir'])
CALIBRATION_BLOCK_PATH = (
    PROCESSED_DIR
    / (SITE_SLUG + '_calibration_' + str(CALIBRATION_YEARS[0])
       + '_' + str(CALIBRATION_YEARS[1]) + '.nc')
)

# LAI ceiling from the site's own measured seasonal maximum, on the chosen basis.
SITE_LAI_AFTER, SITE_LAI_BEFORE = site_lai(CONVENTION)
LAI_CEILING = SITE_LAI_MULTIPLE * SITE_LAI_BEFORE

PLOT_COLOURS = {
    'prior': OKABE_ITO[0],       # blue
    'posterior': OKABE_ITO[1],   # vermillion
    'truth': OKABE_ITO[2],       # bluish green
    'observed': OKABE_ITO[7],    # black
}

print('config          ', CONFIG_PATH)
print('seed            ', SEED)
print('calibration     ', CALIBRATION_YEARS)
print('convention      ', CONVENTION)
print('site LAI max    ', round(SITE_LAI_BEFORE, 3), '(before thinning),',
      round(SITE_LAI_AFTER, 3), '(after)')
print('LAI ceiling     ', round(LAI_CEILING, 3))
print('hdi_prob        ', HDI_PROB)
print()
for label, path in (
    ('calibration block', CALIBRATION_BLOCK_PATH),
    ('true parameters  ', TRUE_PARAMS_PATH),
    ('synthetic block  ', SYNTHETIC_BLOCK_PATH),
    ('trace            ', TRACE_PATH),
    ('posterior pred.  ', SYNTHETIC_POSTPRED_PATH),
):
    print(label, 'present' if Path(path).exists() else 'MISSING ', path)

---

# 1. Prior table

One row per parameter. The numeric bounds are read from the code; the symbol, unit, citation and provenance category are typed in this notebook, because they are recorded in prose in `DECISIONS.md` and in module docstrings rather than in a machine-readable table.

**Provenance takes one of four values:**

- `site measurement` -- a number measured at FI-Hyy or SMEAR II, or derived arithmetically from such a number.
- `published for Scots pine or boreal` -- published, and for this species or this biome.
- `generic DALEC` -- the published DALEC2 range, not specific to this site.
- `guess` -- a judgement with no source behind it.

Where a source cannot be pointed to, the source column says `UNVERIFIED` and a `# TODO` sits next to it. Those are the rows to fix before the thesis quotes them.

**Not every parameter is sampled.** Three groups are built rather than drawn:

1. The four allocation fractions come from a Dirichlet on a 4-simplex, scaled by `1 - f_auto`. Formula: `(f_lab, f_fol, f_roo, f_woo) = (1 - f_auto) * w` where `w` is a 4-vector summing to one. In words: whatever GPP is not respired away is split four ways, and the split is drawn as a proportion so it can never go negative.
2. All six initial carbon pools are derived from a measured flux and a turnover rate, not sampled. Formula for foliage: `C_fol(0) = litterfall / c_lf`. In words: at steady state a pool holds however much is needed for its measured annual loss to match its measured annual input.
3. Heterotrophic respiration is reparameterised: `rh_annual` and `f_som` are sampled and the litter and soil stocks are derived from them, rather than the stocks being sampled directly.

In [ ]:
# Bounds, read from their recorded sources. Nothing numeric is typed here.
SAMPLED_BOUNDS = prior_sources(CONVENTION)          # every sampled scalar
CANOPY_BOUNDS = canopy_bounds(CONVENTION)           # lma, c_lf, ceff overrides
RESPIRATION_BOUNDS = reparameterised_bounds()       # rh_annual, f_som, thetas
DIRICHLET_CONCENTRATION = allocation_concentration()

print('sampled scalars:', len(SAMPLED_BOUNDS))
for name, (low, high) in SAMPLED_BOUNDS.items():
    print('  ' + name.ljust(22), format(low, '.6g').rjust(12),
          format(high, '.6g').rjust(12))
print()
print('Dirichlet concentration over', ALLOCATION_WEIGHT_ORDER)
print(' ', np.round(DIRICHLET_CONCENTRATION, 4),
      ' total', round(float(DIRICHLET_CONCENTRATION.sum()), 4))

In [ ]:
# Provenance metadata. Typed here because it exists only as prose elsewhere.
#
# Fields per parameter:
#   symbol      as written in the source paper where there is one
#   role        sampled | derived | simplex
#   motivation  the numeric value or range that motivated the prior
#   source      citation with year, and table/figure/page where recorded
#   provenance  one of the four categories
#
# Sources are as recorded in DECISIONS.md and in the src/dalec docstrings. Where
# the repository does not record a citation, the source reads UNVERIFIED.

PRIOR_PROVENANCE = {
    'f_auto': dict(
        symbol='f_aut', role='sampled',
        motivation='published range 0.3-0.7; the site fluxes independently imply about 0.51',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'f_lab': dict(
        symbol='f_lab', role='simplex',
        motivation='needle litterfall 154 g C m-2 yr-1, split equally with f_fol',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'f_fol': dict(
        symbol='f_fol', role='simplex',
        motivation='needle litterfall 154 g C m-2 yr-1, split equally with f_lab',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'f_roo': dict(
        symbol='f_roo', role='simplex',
        motivation='below-ground tree litter about 90 g C m-2 yr-1',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'f_woo': dict(
        symbol='f_woo', role='simplex',
        motivation='above-ground net growth 180-240 plus below-ground 34-69, taken as 261',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'theta_roo': dict(
        symbol='theta_roo', role='sampled',
        motivation='published range',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'theta_woo': dict(
        symbol='theta_woo', role='sampled',
        motivation='published range',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'theta_lit': dict(
        symbol='theta_lit', role='sampled',
        motivation='litter residence time 1-5 yr against respiration alone, bracketed by the Yasso07 AWEN rates 0.66, 4.3, 0.35, 0.22 a-1',
        source='Tuomi et al. (2009), Ecological Modelling 220, 3362-3371, Table 1',
        provenance='published for Scots pine or boreal'),
    'theta_som': dict(
        symbol='theta_som', role='sampled',
        motivation='SOM residence time 24-61 yr, solved from soil carbon 6560 g C m-2 and Rh 290-370 g C m-2 yr-1',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6, through parameters.som_residence_time_from_stock',
        provenance='site measurement'),
    'theta_min': dict(
        symbol='theta_min', role='sampled',
        motivation='published range; a litter-to-SOM transfer, not a respiratory loss',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'temperature_exponent': dict(
        symbol='Theta', role='sampled',
        motivation='published range; the FI-Hyy winter flux record gives 0.0366 for reference',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'd_onset': dict(
        symbol='d_onset', role='sampled',
        motivation='published range, the whole year',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'cr_onset': dict(
        symbol='c_ronset', role='sampled',
        motivation='published range',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'd_fall': dict(
        symbol='d_fall', role='sampled',
        motivation='published range, the whole year. NOTE: inert as transcribed -- see README',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'cr_fall': dict(
        symbol='c_rfall', role='sampled',
        motivation='published range',
        source='Bloom and Williams (2015), Biogeosciences 12, 1299-1315, Table 1',
        provenance='generic DALEC'),
    'c_lf': dict(
        symbol='c_lf', role='sampled',
        motivation='reciprocal of a 3-5 year needle longevity for Scots pine in southern Finland',
        source='UNVERIFIED -- the 3-5 year longevity is asserted in DECISIONS.md section 8 with no citation. Candidate to check: Kolari et al. (2009), Boreal Env. Res. 14, 761-783, recorded in DECISIONS.md as giving 25 per cent foliage turnover',
        provenance='published for Scots pine or boreal'),
    'lma': dict(
        symbol='LMA', role='sampled',
        motivation='mean needle litterfall 154 g C m-2 yr-1 times longevity, over the site LAI on the chosen basis',
        source='litterfall: Ilvesniemi et al. (2009), Fig. 6. LAI: Kolari (2010), Dissertationes Forestales 99, doi 10.14214/df.99, all-sided seasonal maximum 6.5-8.0. Conversion: Chen and Black (1992), Plant Cell Environ 15, 421-429 for hemisurface; Niinemets et al. (2001) and Grace (1987), NZ J For Sci 17, 292-294 for projected. Longevity UNVERIFIED as for c_lf',
        provenance='site measurement'),
    'ceff': dict(
        symbol='a1', role='sampled',
        motivation='published range for the same ACM parameter in the REFLEX intercomparison, adopted in place of the wider DALEC2 U(10, 100)',
        source='Fox et al. (2009), Agric. For. Meteorol. 149, 1597-1615, Table 4. DECISIONS.md records this paper as not read in full',
        provenance='generic DALEC'),
    'rh_annual': dict(
        symbol='Rh_annual', role='sampled',
        motivation='290-370 g C m-2 yr-1: soil CO2 efflux 577-737 split about half and half between root plus rhizosphere respiration and decomposition, on a girdling experiment at the site',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'f_som': dict(
        symbol='f_som', role='sampled',
        motivation='SOM share of heterotrophic respiration taken as 0.5-0.9',
        source='UNVERIFIED -- src/dalec/parameters.py states this as a judgement, not a measurement',
        provenance='guess'),
    'c_lab_0': dict(
        symbol='C_lab(0)', role='derived',
        motivation='litterfall 154 times f_lab / (f_lab + f_fol): one year of labile accumulation at the 1 January block start',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'c_fol_0': dict(
        symbol='C_fol(0)', role='derived',
        motivation='litterfall 154 divided by c_lf',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'c_roo_0': dict(
        symbol='C_roo(0)', role='derived',
        motivation='below-ground litterfall 90 over theta_roo times 365.25. Measured living fine-root carbon 238 g C m-2 is held back as an independent check',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6 and Table 4',
        provenance='site measurement'),
    'c_woo_0': dict(
        symbol='C_woo(0)', role='derived',
        motivation='tree carbon 6800 g C m-2 less the labile, foliar and fine-root pools',
        source='Ilvesniemi et al. (2009), Boreal Env. Res. 14(4), 731-753, Fig. 6',
        provenance='site measurement'),
    'c_lit_0': dict(
        symbol='C_lit(0)', role='derived',
        motivation='(1 - f_som) times rh_ref over theta_lit: the litter stock that respires its share of rh_ref at 0 degC',
        source='derived from rh_annual and f_som; see Ilvesniemi et al. (2009) Fig. 6 for rh_annual',
        provenance='site measurement'),
    'c_som_0': dict(
        symbol='C_som(0)', role='derived',
        motivation='f_som times rh_ref over theta_som',
        source='derived from rh_annual and f_som; see Ilvesniemi et al. (2009) Fig. 6 for rh_annual',
        provenance='site measurement'),
}

missing = sorted(set(PARAMETER_NAMES) - set(PRIOR_PROVENANCE))
extra = sorted(set(PRIOR_PROVENANCE) - set(PARAMETER_NAMES) - {'rh_annual', 'f_som'})
print('parameters with no provenance row:', missing or 'none')
print('provenance rows that are not model parameters:', extra or 'none')

In [ ]:
# Assemble and print the prior table. One row per parameter, plus the two
# sampled quantities that are not themselves model parameters (rh_annual,
# f_som) and the Dirichlet weights.

def describe_prior(name, info):
    """The distribution and its settings, as a string, read from the bounds."""
    if info['role'] == 'simplex':
        index = ALLOCATION_WEIGHT_ORDER.index(name)
        alpha = float(DIRICHLET_CONCENTRATION[index])
        return ('Dirichlet component ' + str(index) + ', alpha='
                + format(alpha, '.4g') + ', scaled by (1 - f_auto)')
    if name in SAMPLED_BOUNDS:
        low, high = SAMPLED_BOUNDS[name]
        return 'Uniform(' + format(low, '.6g') + ', ' + format(high, '.6g') + ')'
    return 'derived, not sampled'


def registry_unit(name):
    entry = PARAMETER_REGISTRY.get(name)
    if entry is not None:
        return entry.unit
    return {'rh_annual': 'g C m-2 yr-1', 'f_som': '1'}.get(name, 'UNVERIFIED')


rows = []
for name, info in PRIOR_PROVENANCE.items():
    rows.append({
        'parameter': name,
        'symbol': info['symbol'],
        'unit': registry_unit(name),
        'prior': describe_prior(name, info),
        'motivating value or range': info['motivation'],
        'source': info['source'],
        'provenance': info['provenance'],
        'role': info['role'],
    })

PRIOR_TABLE = pd.DataFrame(rows)

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)
PRIOR_TABLE[['parameter', 'symbol', 'unit', 'prior', 'provenance', 'role']]

In [ ]:
# The same table with the motivation and the citation, which are too wide to
# read alongside the numbers. Print in full rather than truncated.
pd.set_option('display.max_colwidth', None)
for _, row in PRIOR_TABLE.iterrows():
    print(row['parameter'], ' [', row['provenance'], '|', row['role'], ']')
    print('    prior     ', row['prior'])
    print('    motivation', row['motivating value or range'])
    print('    source    ', row['source'])
    print()
pd.set_option('display.max_colwidth', 70)

In [ ]:
# Counts by provenance, and the rows still carrying UNVERIFIED.
print(PRIOR_TABLE['provenance'].value_counts().to_string())
print()
unverified = PRIOR_TABLE[PRIOR_TABLE['source'].str.contains('UNVERIFIED')]
print('rows with an unverified source:', len(unverified))
for name in unverified['parameter']:
    print('  ', name)

### Assumptions behind the prior table, and what is still open

**Assumptions this notebook made in building the table.**

1. Provenance categories were assigned by me from what `DECISIONS.md` and the module docstrings record. Three assignments are arguable and you may want to move them:
   - `ceff` is filed as `generic DALEC` because the REFLEX range is a published prior for the same ACM parameter, not a FI-Hyy measurement. If the REFLEX site is close enough to boreal Scots pine to justify it, move the row. `DECISIONS.md` records Fox et al. (2009) as **not read in full**, so the site is not established here. **# TODO: decide, and record the reason.**
   - `theta_lit` is filed as `published for Scots pine or boreal` because Yasso07 is parameterised on a global litter-bag dataset and is the Finnish soil carbon model, but the rate is a model parameter rather than a FI-Hyy measurement.
   - `theta_som` is filed as `site measurement` although no published SOM residence time is used at all: the range is solved from the site's own soil carbon stock and respiration. It is a derivation from two site measurements, not a measurement.

2. The `unit` column comes from the parameter registry. For `rh_annual` and `f_som`, which are not registry entries, the units are typed in this notebook.

3. The Bloom and Williams table number is recorded as Table 1 throughout the codebase; page numbers within that table are not recorded. **# TODO: add the page if the thesis cites at page level.**

**What is unverified.** The cell above lists the rows. As written they are:

- `c_lf`, and therefore `lma`, which is built from the same longevity. The 3 to 5 year needle longevity is asserted without a citation. It is not a small dependency: it sets the foliar pool, which sets the canopy, which sets GPP.
- `f_som`, which the code itself calls a judgement.

**One thing to check before quoting `lma`.** The `lma` bound depends on `CONVENTION`. Under `hemisurface` it is about 116 to 193 g C m-2 and under `projected` about 148 to 247. The table above shows whichever you set in the config cell. Neither convention is adopted in this project; both are run and the difference reported.

---

# 2. Prior predictive check

A **prior predictive check** draws parameter sets from the priors alone, with no data involved, runs the forward model on each, and looks at what fluxes come out. It asks one question: does what I believe before seeing the data already imply an impossible forest?

It is run on the real FI-Hyy drivers, so the only thing being varied is the parameters.

The recorded reason for doing this is that it has already caught something once: on the generic DALEC2 priors, 837 of 1000 draws produced unusable output, which is what drove the reparameterisation of heterotrophic respiration and the site-informed canopy bounds. **The priors sampled below are the reparameterised ones, not the generic ones**, so the failure rate you get here is not comparable with the 83.7 per cent figure -- it is a different prior.

In [ ]:
# Load the processed calibration block, and optionally cut it to a shorter
# span of years so the first pass is quick.
import dataclasses

if not CALIBRATION_BLOCK_PATH.exists():
    raise FileNotFoundError(
        'no processed calibration block at ' + str(CALIBRATION_BLOCK_PATH)
        + ' -- run scripts/01_prepare_data.py first'
    )

FULL_BLOCK = SiteData.load(CALIBRATION_BLOCK_PATH)


def slice_years(block, years):
    """Return the sub-block covering an inclusive (start, end) year range.

    Every array is cut identically. Nothing is re-derived: the initial pools a
    parameter draw carries are still applied at the first day of whatever block
    is handed to the model, so a shortened block starts the integration at a
    different date with the same initial state.
    """
    if years is None:
        return block
    calendar_years = pd.DatetimeIndex(block.time).year.to_numpy()
    keep = (calendar_years >= years[0]) & (calendar_years <= years[1])
    if not keep.any():
        raise ValueError('no day of the block falls in ' + str(years))
    arrays = {}
    for field in dataclasses.fields(block):
        value = getattr(block, field.name)
        if isinstance(value, np.ndarray):
            arrays[field.name] = value[keep]
    arrays['partitioned'] = {k: v[keep] for k, v in block.partitioned.items()}
    return dataclasses.replace(block, **arrays)


BLOCK = slice_years(FULL_BLOCK, PRIOR_PRED_YEARS)
BLOCK_YEARS = pd.DatetimeIndex(BLOCK.time).year.to_numpy()
BLOCK_DATES = pd.DatetimeIndex(BLOCK.time)

print('full block      ', FULL_BLOCK.n_days, 'days,',
      FULL_BLOCK.n_assimilated, 'assimilable')
print('block in use    ', BLOCK.n_days, 'days,', BLOCK.n_assimilated,
      'assimilable,', BLOCK_DATES[0].date(), 'to', BLOCK_DATES[-1].date())
print('years present   ', sorted(set(BLOCK_YEARS.tolist())))
print('drivers finite  ', all(np.isfinite(v).all() for v in BLOCK.drivers().values()))
print('t_max >= t_min  ', bool((BLOCK.t_max >= BLOCK.t_min).all()))

In [ ]:
# Draw parameter sets from the prior. No data is involved.
#
# sample_reparameterised_parameters draws the sampled scalars, the Dirichlet
# allocation weights, and derives every initial pool. It uses the full block's
# air temperature, not the sliced one, because rh_ref inverts an annual
# respiration total through the mean temperature multiplier and that total is
# stated over the whole record.
ACM = acm_from_config(CONFIG)
RNG = np.random.default_rng(SEED)

started = time.perf_counter()
PRIOR_PARAMS, PRIOR_FRAME = sample_reparameterised_parameters(
    N_PRIOR_DRAWS, rng=RNG, t_air=FULL_BLOCK.t_air
)
print('drew', len(PRIOR_PARAMS), 'parameter sets in',
      round(time.perf_counter() - started, 2), 's')
print('acm:', ACM)
print()
PRIOR_FRAME.describe().T[['mean', 'std', 'min', '50%', 'max']]

In [ ]:
# Run the forward model once per draw and store the fluxes.
#
# Stored per draw: the daily NEE series, the daily GPP series if STORE_FULL_GPP,
# the minimum each carbon pool reaches, the maximum LAI, and the label
# classify_prior_draw gives the draw. Full pool trajectories are not kept --
# they are six times the size of NEE and nothing below needs them.
#
# LAI = C_fol / lma. In words: leaf area follows from the foliar carbon a draw
# grows and the leaf mass per unit area it was given.

n_days = BLOCK.n_days
n_draws = len(PRIOR_PARAMS)

prior_run = {
    'nee': np.full((n_draws, n_days), np.nan),
    'gpp': np.full((n_draws, n_days), np.nan) if STORE_FULL_GPP else None,
    'gpp_min': np.full(n_draws, np.nan),
    'pool_min': np.full((n_draws, len(POOL_NAMES)), np.nan),
    'lai_max': np.full(n_draws, np.nan),
    'ran': np.zeros(n_draws, dtype=bool),
    'label': [],
    'error': [],
}

started = time.perf_counter()
for index, params in enumerate(PRIOR_PARAMS):
    try:
        output = run_dalec2(params, BLOCK, gpp_fn=ACM)
    except Exception as exc:                       # noqa: BLE001 - recorded, not hidden
        prior_run['label'].append('raised')
        prior_run['error'].append(type(exc).__name__ + ': ' + str(exc)[:120])
        continue
    prior_run['ran'][index] = True
    prior_run['nee'][index] = output.nee
    if STORE_FULL_GPP:
        prior_run['gpp'][index] = output.gpp
    prior_run['gpp_min'][index] = output.gpp.min()
    prior_run['pool_min'][index] = output.pools.min(axis=0)
    lai = leaf_area_index(output.pool('c_fol'), params.lma)
    prior_run['lai_max'][index] = float(np.max(lai))
    prior_run['label'].append(classify_prior_draw(output) or 'ok')
    prior_run['error'].append('')
    if (index + 1) % PROGRESS_EVERY == 0:
        print('  draw', index + 1, 'of', n_draws, ' elapsed',
              round(time.perf_counter() - started, 1), 's', flush=True)

prior_run['elapsed_s'] = time.perf_counter() - started
print('done in', round(prior_run['elapsed_s'], 1), 's')
print('draws that raised:', int((~prior_run['ran']).sum()))
print()
print('classify_prior_draw labels, from dalec.diagnostics:')
print(pd.Series(prior_run['label']).value_counts().to_string())
print()
print('ACM counters, accumulated over ALL draws (not per draw):')
print('  discriminant clamps', ACM.clamp_count)
print('  temperature range floored', ACM.range_floor_count)

### 2a. Fan plot: simulated daily NEE across one calibration year

What the next cell draws, and nothing more:

- **Shaded bands**: the 5th to 95th and 25th to 75th percentiles of simulated daily NEE *across prior draws*, one percentile set per calendar day of `FAN_YEAR`.
- **Solid coloured line**: the median across draws, same days.
- **Grey band**: the observed NEE reference range. For each day of year, the `OBS_BAND_PERCENTILES` range of the measured NEE across every year in the block, using assimilable days only. With the default `(0, 100)` this is the observed minimum to maximum.
- **Black line**: the measured NEE in `FAN_YEAR` itself, assimilable days only.

Sign convention is FLUXNET's: negative NEE is net uptake by the ecosystem.

The y-axis is clipped to `FAN_YLIM` so that one extreme draw cannot flatten the rest. The count of series leaving the axis is printed, so you know how much is off-screen.

In [ ]:
# y-limits for the fan plot. Set to None to let matplotlib autoscale.
FAN_YLIM = (-12.0, 12.0)

fan_index = np.flatnonzero(BLOCK_YEARS == FAN_YEAR)
if fan_index.size == 0:
    raise ValueError(
        'FAN_YEAR ' + str(FAN_YEAR) + ' is not in the block; years present are '
        + str(sorted(set(BLOCK_YEARS.tolist())))
    )

# Only draws that integrated and stayed finite across this year can be
# percentiled; a single NaN would take the whole band out.
usable = prior_run['ran'] & np.isfinite(prior_run['nee'][:, fan_index]).all(axis=1)
print('draws in the fan:', int(usable.sum()), 'of', n_draws)

fan_dates = BLOCK_DATES[fan_index]
fan_doy = BLOCK.doy[fan_index]
fan_nee = prior_run['nee'][usable][:, fan_index]
fan_bands = np.percentile(fan_nee, FAN_PERCENTILES, axis=0)

# Observed reference band by day of year, over the whole block.
observed = pd.DataFrame({
    'doy': BLOCK.doy,
    'nee': np.where(BLOCK.nee_mask, BLOCK.nee_obs, np.nan),
})
by_doy = observed.groupby('doy')['nee']
obs_low = by_doy.quantile(OBS_BAND_PERCENTILES[0] / 100.0).reindex(fan_doy).to_numpy()
obs_high = by_doy.quantile(OBS_BAND_PERCENTILES[1] / 100.0).reindex(fan_doy).to_numpy()
obs_year = np.where(BLOCK.nee_mask[fan_index], BLOCK.nee_obs[fan_index], np.nan)

figure, axis = plt.subplots(figsize=(11, 5))
axis.fill_between(fan_dates, obs_low, obs_high, color=PLOT_COLOURS['observed'],
                  alpha=0.18, zorder=1,
                  label='observed range by day of year, whole block')
axis.fill_between(fan_dates, fan_bands[0], fan_bands[4], color=PLOT_COLOURS['prior'],
                  alpha=0.25, zorder=2, label='prior 5-95 per cent')
axis.fill_between(fan_dates, fan_bands[1], fan_bands[3], color=PLOT_COLOURS['prior'],
                  alpha=0.45, zorder=3, label='prior 25-75 per cent')
axis.plot(fan_dates, fan_bands[2], color=PLOT_COLOURS['prior'], lw=1.6, zorder=4,
          label='prior median')
axis.plot(fan_dates, obs_year, color=PLOT_COLOURS['observed'], lw=1.4, zorder=5,
          label='observed NEE, ' + str(FAN_YEAR))
axis.axhline(0.0, color='0.4', lw=0.8, zorder=0)
axis.set_ylabel('NEE, g C m$^{-2}$ d$^{-1}$')
axis.set_title('Prior predictive NEE, ' + str(FAN_YEAR) + ' -- '
               + str(fan_nee.shape[0]) + ' draws, ' + CONVENTION + ' convention')
axis.legend(ncol=2, loc='upper left')
if FAN_YLIM is not None:
    axis.set_ylim(*FAN_YLIM)
    outside = int((np.nanmax(np.abs(fan_nee), axis=1)
                   > max(abs(v) for v in FAN_YLIM)).sum())
    print('draws leaving the y-axis:', outside, 'of', fan_nee.shape[0])
plt.show()

### 2b. Physical plausibility rules

Each rule is a separate named function taking the stored run and returning one boolean per draw: `True` means that draw **fails** that rule. They are independent -- a draw can fail several.

| Rule | What it tests | Threshold source |
|---|---|---|
| `integration_raised` | the forward model raised an exception | not a threshold |
| `non_finite_flux` | any NaN or infinity in daily NEE | not a threshold |
| `gpp_below_zero` | minimum daily GPP below `GPP_FLOOR_G_C_M2_D` | physical: gross uptake cannot be negative |
| `lai_above_ceiling` | maximum LAI above `LAI_CEILING` | `SITE_LAI_MULTIPLE` times the site's measured seasonal maximum, on the chosen basis |
| `pool_negative` | any of the six carbon pools going below zero on any day | physical: a carbon stock cannot be negative |
| `annual_nee_outside_range` | any complete calendar year with annual NEE outside `ANNUAL_NEE_RANGE_G_C_M2_YR` | stated assumption, set in the config cell |

Annual NEE is the plain sum of daily NEE over each calendar year with at least `MIN_DAYS_FOR_ANNUAL` days in the block. Formula: `annual_NEE(y) = sum over days in y of NEE(d)`. In words: add up the daily net exchange to get the year's net carbon balance, negative being a sink.

**Draws that did not integrate are not evaluated by the other rules.** They are counted once, in `integration_raised`, and the `not evaluable` column of the table says how many draws each rule could not be applied to.

In [ ]:
# Annual NEE per draw per complete year, and the evaluable mask.
complete_years = [
    int(year) for year in sorted(set(BLOCK_YEARS.tolist()))
    if int((BLOCK_YEARS == year).sum()) >= MIN_DAYS_FOR_ANNUAL
]
if not complete_years:
    raise ValueError(
        'no calendar year in the block has at least ' + str(MIN_DAYS_FOR_ANNUAL)
        + ' days; lower MIN_DAYS_FOR_ANNUAL or widen PRIOR_PRED_YEARS'
    )

prior_run['annual_nee'] = np.column_stack([
    prior_run['nee'][:, BLOCK_YEARS == year].sum(axis=1) for year in complete_years
])
prior_run['finite'] = np.isfinite(prior_run['nee']).all(axis=1)
prior_run['evaluable'] = prior_run['ran'] & prior_run['finite']

print('complete years:', complete_years)
print('draws evaluable by the physical rules:',
      int(prior_run['evaluable'].sum()), 'of', n_draws)
print()
print('annual NEE across draws and years, g C m-2 yr-1:')
annual = prior_run['annual_nee'][prior_run['evaluable']]
print(pd.DataFrame(annual, columns=[str(y) for y in complete_years])
      .describe().T[['mean', 'std', 'min', '50%', 'max']].to_string())

In [ ]:
# The rules, one named function each. True means the draw fails.

def rule_integration_raised(run):
    return ~run['ran']


def rule_non_finite_flux(run):
    return run['ran'] & ~run['finite']


def rule_gpp_below_zero(run):
    return (run['gpp_min'] < GPP_FLOOR_G_C_M2_D) & run['evaluable']


def rule_lai_above_ceiling(run):
    return (run['lai_max'] > LAI_CEILING) & run['evaluable']


def rule_pool_negative(run):
    return (run['pool_min'].min(axis=1) < 0.0) & run['evaluable']


def rule_annual_nee_outside_range(run):
    low, high = ANNUAL_NEE_RANGE_G_C_M2_YR
    outside = (run['annual_nee'] < low) | (run['annual_nee'] > high)
    return outside.any(axis=1) & run['evaluable']


RULES = {
    'integration_raised': rule_integration_raised,
    'non_finite_flux': rule_non_finite_flux,
    'gpp_below_zero': rule_gpp_below_zero,
    'lai_above_ceiling': rule_lai_above_ceiling,
    'pool_negative': rule_pool_negative,
    'annual_nee_outside_range': rule_annual_nee_outside_range,
}

# Which draws each rule could be applied to at all.
RULE_SCOPE = {
    'integration_raised': np.ones(n_draws, dtype=bool),
    'non_finite_flux': prior_run['ran'],
    'gpp_below_zero': prior_run['evaluable'],
    'lai_above_ceiling': prior_run['evaluable'],
    'pool_negative': prior_run['evaluable'],
    'annual_nee_outside_range': prior_run['evaluable'],
}

RULE_FAILURES = {name: rule(prior_run) for name, rule in RULES.items()}
ANY_FAILURE = np.zeros(n_draws, dtype=bool)
for flags in RULE_FAILURES.values():
    ANY_FAILURE |= flags

print('rules defined:', list(RULES))

In [ ]:
# The rule table: name, count of failing draws, percentage.
rule_rows = []
for name, flags in RULE_FAILURES.items():
    scope = RULE_SCOPE[name]
    rule_rows.append({
        'rule': name,
        'failing draws': int(flags.sum()),
        'per cent of all draws': round(100.0 * flags.sum() / n_draws, 2),
        'not evaluable': int((~scope).sum()),
        'per cent of evaluable': (
            round(100.0 * flags.sum() / scope.sum(), 2) if scope.sum() else float('nan')
        ),
    })
rule_rows.append({
    'rule': 'ANY RULE',
    'failing draws': int(ANY_FAILURE.sum()),
    'per cent of all draws': round(100.0 * ANY_FAILURE.sum() / n_draws, 2),
    'not evaluable': 0,
    'per cent of evaluable': float('nan'),
})

RULE_TABLE = pd.DataFrame(rule_rows)
print('draws:', n_draws, ' block:', BLOCK.n_days, 'days,',
      BLOCK_DATES[0].date(), 'to', BLOCK_DATES[-1].date())
print('LAI ceiling', round(LAI_CEILING, 3),
      ' annual NEE window', ANNUAL_NEE_RANGE_G_C_M2_YR)
RULE_TABLE

### 2c. Which prior is driving each failure

For every rule that some draws fail and some pass, the next cell compares the failing draws against the passing ones, one parameter at a time. Three numbers per parameter:

- **`ks`** -- the two-sample Kolmogorov-Smirnov statistic: the largest vertical gap between the cumulative distribution of that parameter among the failing draws and among the passing draws. It runs from 0 to 1. In words: how far apart the two groups sit on that one parameter's axis, ignoring every other parameter.
- **`p`** -- the KS test's p-value, reported unrounded. It is a test of whether the two samples could come from the same distribution.
- **`median_fail`, `median_pass`** -- the median of that parameter within each group, so the direction of the shift is visible.

Also reported per parameter is the failure rate in the bottom and top fifth of that parameter's own sampled range, which shows whether the failures pile up at one end of the prior.

What this cannot do: the statistic is univariate. A failure caused by a *combination* of two parameters, neither of which is unusual on its own, will show weakly or not at all on both.

In [ ]:
from scipy.stats import ks_2samp

# Parameters to break down. Everything in the draws table except the index.
BREAKDOWN_PARAMETERS = [c for c in PRIOR_FRAME.columns if c != 'draw']


def breakdown_for_rule(name, flags, scope):
    """Per-parameter separation between the failing and passing draws."""
    failing = flags & scope
    passing = (~flags) & scope
    if failing.sum() == 0 or passing.sum() == 0:
        return None
    rows = []
    for column in BREAKDOWN_PARAMETERS:
        values = PRIOR_FRAME[column].to_numpy()
        fail_values, pass_values = values[failing], values[passing]
        test = ks_2samp(fail_values, pass_values)
        in_scope = values[scope]
        low_cut, high_cut = np.percentile(in_scope, [20.0, 80.0])
        bottom = scope & (values <= low_cut)
        top = scope & (values >= high_cut)
        rows.append({
            'parameter': column,
            'ks': test.statistic,
            'p': test.pvalue,
            'median_fail': float(np.median(fail_values)),
            'median_pass': float(np.median(pass_values)),
            'fail_rate_bottom_fifth': (
                float(flags[bottom].mean()) if bottom.sum() else float('nan')),
            'fail_rate_top_fifth': (
                float(flags[top].mean()) if top.sum() else float('nan')),
        })
    frame = pd.DataFrame(rows).sort_values('ks', ascending=False)
    frame.insert(0, 'rule', name)
    return frame.reset_index(drop=True)


BREAKDOWNS = {}
for name, flags in list(RULE_FAILURES.items()) + [('ANY RULE', ANY_FAILURE)]:
    scope = RULE_SCOPE.get(name, np.ones(n_draws, dtype=bool))
    result = breakdown_for_rule(name, flags, scope)
    if result is None:
        print(name, '-- no split: ', int((flags & scope).sum()), 'failing of',
              int(scope.sum()), 'in scope')
    else:
        BREAKDOWNS[name] = result
print()
print('breakdowns available for:', list(BREAKDOWNS))

In [ ]:
# Top parameters per rule, by KS statistic. Change N_SHOW to see more.
N_SHOW = 8

for name, frame in BREAKDOWNS.items():
    print('=' * 72)
    print(name, ' -- ', int(RULE_FAILURES.get(name, ANY_FAILURE).sum()),
          'failing draws')
    print('=' * 72)
    print(frame.head(N_SHOW).to_string(index=False))
    print()

In [ ]:
# The same information as one picture: KS statistic per parameter per rule.
if BREAKDOWNS:
    wide = pd.concat(BREAKDOWNS.values()).pivot(
        index='parameter', columns='rule', values='ks')
    wide = wide.loc[wide.max(axis=1).sort_values(ascending=False).index]
    figure, axis = plt.subplots(figsize=(1.6 * wide.shape[1] + 4, 0.30 * len(wide) + 2))
    image = axis.imshow(wide.to_numpy(), aspect='auto', cmap='magma_r',
                        vmin=0.0, vmax=1.0)
    axis.set_xticks(range(wide.shape[1]))
    axis.set_xticklabels(wide.columns, rotation=30, ha='right')
    axis.set_yticks(range(len(wide)))
    axis.set_yticklabels(wide.index)
    axis.set_title('KS separation between failing and passing draws')
    axis.grid(False)
    figure.colorbar(image, ax=axis, label='KS statistic')
    plt.show()
else:
    print('no rule produced both failing and passing draws, so there is nothing to plot')

---

# 3. Branches: where each prior-check outcome leads

These are the choices in front of you once section 2 has run, written as branches. Which branch applies is your reading of the numbers, not mine. More than one can apply at once.

**Branch A -- almost every draw passes every rule, and the NEE fan covers the observed band without dwarfing it.**
The priors are not, on their own, claiming an impossible forest. Next step: go to the synthetic stage, section 4. Record the pass rate in the thesis as the evidence, with the draw count.

**Branch B -- a small but non-zero share fails, concentrated in one parameter by the section 2c breakdown.**
The choice is between three things, and they are not equivalent:
- Tighten that parameter's prior, if a source supports the tighter range. This changes a locked decision and needs the source first.
- Leave the prior alone and record the failure rate as a measured property of it. This is what the project did for the generic priors: the 83.7 per cent was reported, not fixed away.
- Check whether the rule threshold, rather than the prior, is what is wrong -- particularly for `lai_above_ceiling` and `annual_nee_outside_range`, both of which are assumptions typed in the config cell of this notebook.

**Branch C -- a large share fails, spread across many parameters with no single one separating.**
A univariate breakdown will not find a joint cause. Options: look at the failing draws' *derived* quantities rather than the sampled ones (`rh_ref`, `c_fol_0`, `c_woo_0` are in the same table); or check whether the failures are a single mechanism by plotting the NEE of the failing draws alone.

**Branch D -- draws fail by raising, not by producing bad numbers.**
Read the `error` column of `prior_run`. One specific case is known: `sample_reparameterised_parameters` raises for the *entire batch* if any single draw drives the derived wood carbon non-positive, which happens when `theta_roo` reaches the slow end of its range. That is a batch failure at draw time, not at run time, and it means no draws at all.

**Branch E -- the fan is far wider than the observed band, but nothing fails a rule.**
That is not a failure of the prior check; it is what a weak prior looks like. It does bear on what section 6 and 7 can show: a prior this wide leaves room for the data to move the posterior, so parameters that then do not move are candidates for being unidentifiable.

**Branch F -- the fan does not cover the observed band at all.**
The prior predictive and the observations disagree before any fitting. Next step is to find out which flux is responsible, which is what `dalec.diagnostics.gpp_magnitude_gate` and `annual_gpp_comparison` exist for; they are not run in this notebook.

**Branch G -- the result changes materially between 200 draws and 1000.**
Raise `N_PRIOR_DRAWS` and re-run. Remember the draws are not nested: 1000 draws is a different sample, not the 200 plus 800 more.

---

# 4. The true parameter set and the synthetic dataset

The synthetic stage: generate NEE from the forward model with a known parameter set, add noise, then run the sampler on that. Because the truth is known, the posterior can be checked against it -- which is not possible with real data.

**These files do not exist yet.** `scripts/03_synthetic_twin.py` raises `SystemExit`; Phase 7 is unbuilt. The cells below assume a file contract, and it is an assumption, not a fact:

| Config name | Assumed contents |
|---|---|
| `TRUE_PARAMS_PATH` | JSON object, one key per `DalecParameters` field, 24 keys. A nested `{'parameters': {...}}` is also accepted. |
| `SYNTHETIC_BLOCK_PATH` | NetCDF written by `SiteData.save`, identical drivers to the real block, with `nee_obs` replaced by the synthetic series. `nee_unc` is assumed to carry the per-day noise standard deviation actually used. |

**# TODO: when `scripts/03_synthetic_twin.py` is written, change the paths in the config cell and check the two assumptions above against what it really writes.** In particular: whether the noise standard deviation is stored anywhere, and whether the mask is the real `NEE_VUT_REF_QC` mask or an all-true mask. Both change how section 9 should be read.

In [ ]:
# Load the true parameter set.
if not Path(TRUE_PARAMS_PATH).exists():
    raise FileNotFoundError(
        'no true parameter file at ' + str(TRUE_PARAMS_PATH) + '. '
        'scripts/03_synthetic_twin.py is not built yet -- see the markdown above.'
    )

raw_truth = json.loads(Path(TRUE_PARAMS_PATH).read_text(encoding='utf-8'))
if 'parameters' in raw_truth and isinstance(raw_truth['parameters'], dict):
    raw_truth = raw_truth['parameters']

missing_fields = sorted(set(PARAMETER_NAMES) - set(raw_truth))
if missing_fields:
    raise KeyError('true parameter file is missing fields: ' + str(missing_fields))

TRUE_VALUES = {name: float(raw_truth[name]) for name in PARAMETER_NAMES}

# Printed before the container is built: DalecParameters checks the allocation
# closure in __post_init__ and raises, so this is the last chance to see it.
closure = sum(TRUE_VALUES[name] for name in ALLOCATION_PARAMETERS)
print('true parameter set loaded:', len(TRUE_VALUES), 'fields')
print('allocation closure sums to', closure)

TRUE_PARAMS = DalecParameters(**TRUE_VALUES)
print()

truth_table = pd.DataFrame({
    'true value': pd.Series(TRUE_VALUES),
    'prior low': pd.Series({k: v[0] for k, v in SAMPLED_BOUNDS.items()}),
    'prior high': pd.Series({k: v[1] for k, v in SAMPLED_BOUNDS.items()}),
})
truth_table['inside prior'] = (
    (truth_table['true value'] >= truth_table['prior low'])
    & (truth_table['true value'] <= truth_table['prior high'])
)
truth_table

In [ ]:
# Load the synthetic dataset and check its shapes and ranges.
if not Path(SYNTHETIC_BLOCK_PATH).exists():
    raise FileNotFoundError(
        'no synthetic block at ' + str(SYNTHETIC_BLOCK_PATH) + '. '
        'scripts/03_synthetic_twin.py is not built yet -- see the markdown above.'
    )

SYNTH = SiteData.load(SYNTHETIC_BLOCK_PATH)
SYNTH_DATES = pd.DatetimeIndex(SYNTH.time)

print('days                ', SYNTH.n_days)
print('assimilable days    ', SYNTH.n_assimilated)
print('date range          ', SYNTH_DATES[0].date(), 'to', SYNTH_DATES[-1].date())
print('calendar gaps       ', int((np.diff(SYNTH.time).astype('timedelta64[D]')
                                   .astype(int) != 1).sum()))
print('attrs               ', dict(SYNTH.attrs))
print()
ranges = pd.DataFrame({
    name: {
        'min': float(np.nanmin(values)),
        'median': float(np.nanmedian(values)),
        'max': float(np.nanmax(values)),
        'n nan': int(np.isnan(values).sum()),
    }
    for name, values in {
        't_air': SYNTH.t_air, 't_max': SYNTH.t_max, 't_min': SYNTH.t_min,
        'sw_in': SYNTH.sw_in, 'co2': SYNTH.co2,
        'nee_obs': SYNTH.nee_obs, 'nee_unc': SYNTH.nee_unc,
    }.items()
}).T
ranges

In [ ]:
# Does the synthetic block share its drivers with the real block? If the
# synthetic twin was generated over the same years, they should be identical.
# Compared on the overlapping dates only.
shared = SYNTH_DATES.intersection(pd.DatetimeIndex(FULL_BLOCK.time))
print('dates shared with the real calibration block:', len(shared))

if len(shared):
    left = pd.Index(SYNTH_DATES).get_indexer(shared)
    right = pd.Index(pd.DatetimeIndex(FULL_BLOCK.time)).get_indexer(shared)
    for name in ('t_air', 't_max', 't_min', 'sw_in', 'co2'):
        a = getattr(SYNTH, name)[left]
        b = getattr(FULL_BLOCK, name)[right]
        print('  ' + name.ljust(7), 'identical' if np.allclose(a, b) else
          'DIFFERENT, max abs diff ' + format(float(np.nanmax(np.abs(a - b))), '.4g'))
    obs_a = SYNTH.nee_obs[left]
    obs_b = FULL_BLOCK.nee_obs[right]
    both = np.isfinite(obs_a) & np.isfinite(obs_b)
    print('  nee_obs  synthetic vs measured, correlation over',
          int(both.sum()), 'shared finite days:',
          round(float(np.corrcoef(obs_a[both], obs_b[both])[0, 1]), 4)
          if both.sum() > 2 else 'too few days')

---

# 5. The posterior: R-hat and effective sample size

**R-hat** compares the variance between chains with the variance within each chain. If the chains have converged on the same distribution the two agree and R-hat approaches 1. The project's threshold is in `config/default.yaml` as `diagnostics.rhat_threshold`.

**Effective sample size (ESS)** is how many independent draws the correlated chain is worth. `ess_bulk` covers the body of the distribution, `ess_tail` the 5th and 95th percentile region. Both should be read against the total number of post-warmup draws: a short run cannot have a large ESS no matter how well it mixes.

### A rounding trap, and it has already caught this project once

`az.summary(idata, round_to=None)` **does not disable rounding.** ArviZ reads Python `None` as the default and falls through to `rcParams['stats.round_to']`, which is two significant figures. Only the *string* `'none'` disables it.

Verified against the arviz 1.2.0 installed in this repository's `.venv`: `summary()` contains `if round_to == 'auto' or round_to is None: round_val = rcParams['stats.round_to']`, and the rounding is skipped only when `round_to` is one of the strings `'None'` or `'none'`.

What it cost here, recorded in `reports/sampling/FINDINGS_sampling.md`: the plumbing run reported a worst R-hat of 1.000 when the true value was 1.0199. Two variables were over the 1.01 threshold and the table showed none, because 1.0199 and 1.0138 both round to 1.0 at two significant figures. `scripts/20_plumbing_run.py` was fixed; **`scripts/04_calibrate.py:559` still passes `round_to=None`**, so every R-hat and ESS reported from the full calibration is rounded.

The cell below passes `round_to='none'`.

In [ ]:
import arviz as az

if not Path(TRACE_PATH).exists():
    raise FileNotFoundError(
        'no trace at ' + str(TRACE_PATH) + '. Set TRACE_PATH_OVERRIDE in the '
        'config cell to an existing trace, or build the synthetic twin.'
    )

IDATA = az.from_netcdf(str(TRACE_PATH))


def as_dataset(node):
    """ArviZ 1.x returns a DataTree; this gets the Dataset out of a node."""
    return node.to_dataset() if hasattr(node, 'to_dataset') else node.dataset


POSTERIOR = as_dataset(IDATA['posterior'])
N_CHAINS = int(POSTERIOR.sizes['chain'])
N_DRAWS_PER_CHAIN = int(POSTERIOR.sizes['draw'])

print('arviz         ', az.__version__)
print('trace         ', TRACE_PATH)
print('groups        ', IDATA.groups)
print('chains x draws', N_CHAINS, 'x', N_DRAWS_PER_CHAIN,
      '=', N_CHAINS * N_DRAWS_PER_CHAIN, 'post-warmup draws')
print('variables     ', len(POSTERIOR.data_vars))

In [ ]:
# R-hat and ESS per variable, unrounded. round_to='none' is the only value
# that disables rounding -- see the markdown above.
SUMMARY = az.summary(IDATA, round_to='none')

wanted = [c for c in ('r_hat', 'ess_bulk', 'ess_tail', 'mean', 'sd')
          if c in SUMMARY.columns]
CONVERGENCE = SUMMARY[wanted].sort_values('r_hat', ascending=False)

print('rhat threshold from config:', RHAT_THRESHOLD)
print('variables above it        :',
      int((CONVERGENCE['r_hat'] > RHAT_THRESHOLD).sum()), 'of', len(CONVERGENCE))
print('variables above 1.01      :',
      int((CONVERGENCE['r_hat'] > 1.01).sum()))
print('worst r_hat               :', CONVERGENCE['r_hat'].max())
print('lowest ess_bulk           :', CONVERGENCE['ess_bulk'].min())
print('lowest ess_tail           :', CONVERGENCE['ess_tail'].min())
CONVERGENCE

In [ ]:
# Addition, not in the requested list: the sampler's own statistics. Delete
# this cell if you do not want it. R-hat read without the divergence count can
# look healthy on a run that was rejecting trajectories throughout.
if '/sample_stats' in IDATA.groups:
    stats = as_dataset(IDATA['sample_stats'])
    print('sample_stats variables:', list(stats.data_vars))
    if 'diverging' in stats:
        diverging = stats['diverging'].to_numpy()
        print('divergences        ', int(diverging.sum()), 'of', diverging.size)
        print('  per chain        ', diverging.sum(axis=1).tolist())
    if 'tree_depth' in stats:
        depth = stats['tree_depth'].to_numpy()
        print('tree depth median  ', float(np.median(depth)), ' max', int(depth.max()))
    if 'lp' in stats:
        lp = stats['lp'].to_numpy()
        print('mean log posterior per chain:')
        for chain in range(lp.shape[0]):
            print('  chain', chain, format(float(lp[chain].mean()), '.1f'))
else:
    print('no sample_stats group in this trace')

In [ ]:
# Flatten the posterior to a dataframe: one row per draw, one column per
# scalar variable. Vector variables such as allocation_weights become
# allocation_weights[0], [1], ... This is the same flattening
# scripts/23_arviz_diagnostics.py uses.

def posterior_frame(dataset):
    columns = {}
    for name in dataset.data_vars:
        array = dataset[name].transpose('chain', 'draw', ...)
        extra = [d for d in array.dims if d not in ('chain', 'draw')]
        if not extra:
            columns[name] = array.to_numpy().reshape(-1)
        else:
            stacked = array.stack(_flat=extra)
            for index in range(stacked.sizes['_flat']):
                columns[name + '[' + str(index) + ']'] = (
                    stacked.isel(_flat=index).to_numpy().reshape(-1))
    frame = pd.DataFrame(columns)
    frame.insert(0, 'draw', np.tile(np.arange(N_DRAWS_PER_CHAIN), N_CHAINS))
    frame.insert(0, 'chain', np.repeat(np.arange(N_CHAINS), N_DRAWS_PER_CHAIN))
    return frame


POST = posterior_frame(POSTERIOR)
POST_PARAMETERS = [c for c in POST.columns if c not in ('chain', 'draw')]
print('posterior frame:', POST.shape[0], 'draws x', len(POST_PARAMETERS), 'columns')
print(POST_PARAMETERS)

---

# 6. Recovery: the posterior against the known truth

Two things per parameter.

**The plot**: the posterior draws as a histogram, with the true value as a vertical line.

**The rank**: how many posterior draws fall below the true value. Formula: `rank = count of draws with value < truth`, and `quantile = rank / n_draws`. In words: where in its own posterior the truth sits, 0 meaning the truth is below every draw and 1 meaning above every draw.

What a single rank does and does not tell you: it locates the truth within one posterior. It is **not** simulation-based calibration, which needs many synthetic datasets and asks whether the ranks are uniform across them. One rank from one dataset cannot be uniform or not.

Also printed is whether the truth falls inside the equal-tailed interval at `HDI_PROB`, and the posterior's distance from the truth in posterior standard deviations: `z = (mean - truth) / sd`.

In [ ]:
# Which parameters can be compared: present in the posterior and in the truth.
RECOVERABLE = [name for name in PARAMETER_NAMES if name in POST.columns]
NOT_IN_POSTERIOR = [name for name in PARAMETER_NAMES if name not in POST.columns]

print('comparable parameters:', len(RECOVERABLE))
print('in the truth but not in the posterior:', NOT_IN_POSTERIOR or 'none')
print('in the posterior but not a model parameter:',
      [c for c in POST_PARAMETERS if c not in PARAMETER_NAMES])

In [ ]:
# Rank of the true value within the posterior, per parameter.
tail = (1.0 - HDI_PROB) / 2.0
rows = []
for name in RECOVERABLE:
    samples = POST[name].to_numpy()
    truth = TRUE_VALUES[name]
    low, high = np.percentile(samples, [100.0 * tail, 100.0 * (1.0 - tail)])
    sd = float(samples.std(ddof=1))
    rows.append({
        'parameter': name,
        'true': truth,
        'post mean': float(samples.mean()),
        'post sd': sd,
        'rank': int((samples < truth).sum()),
        'n draws': samples.size,
        'quantile': float((samples < truth).mean()),
        'eti low': float(low),
        'eti high': float(high),
        'truth in eti': bool(low <= truth <= high),
        'z': float((samples.mean() - truth) / sd) if sd > 0 else float('nan'),
    })

RECOVERY = pd.DataFrame(rows).set_index('parameter')
print('interval:', HDI_PROB, 'equal-tailed')
print('parameters with the truth inside the interval:',
      int(RECOVERY['truth in eti'].sum()), 'of', len(RECOVERY))
RECOVERY

In [ ]:
# Posterior against the true value, one panel per parameter.
N_COLS = 4
n_rows = int(np.ceil(len(RECOVERABLE) / N_COLS))
figure, axes = plt.subplots(n_rows, N_COLS, figsize=(3.4 * N_COLS, 2.3 * n_rows))
axes = np.atleast_1d(axes).ravel()

for axis, name in zip(axes, RECOVERABLE):
    samples = POST[name].to_numpy()
    axis.hist(samples, bins=40, color=PLOT_COLOURS['posterior'], alpha=0.75,
              density=True)
    axis.axvline(TRUE_VALUES[name], color=PLOT_COLOURS['truth'], lw=2.0)
    quantile = RECOVERY.loc[name, 'quantile']
    axis.set_title(name + '  q=' + format(quantile, '.3f'))
    axis.set_yticks([])
for axis in axes[len(RECOVERABLE):]:
    axis.set_visible(False)
figure.suptitle('Posterior (bars) against the true value (line)')
plt.show()

In [ ]:
# The ranks on one axis, so they can be read together.
figure, axis = plt.subplots(figsize=(9, 0.28 * len(RECOVERY) + 2))
order = RECOVERY['quantile'].sort_values().index
axis.scatter(RECOVERY.loc[order, 'quantile'], range(len(order)),
             color=PLOT_COLOURS['posterior'], zorder=3)
axis.axvline(0.5, color='0.4', lw=0.8)
axis.axvspan(tail, 1.0 - tail, color=PLOT_COLOURS['truth'], alpha=0.10,
             label='central ' + str(HDI_PROB))
axis.set_yticks(range(len(order)))
axis.set_yticklabels(order)
axis.set_xlim(0, 1)
axis.set_xlabel('quantile of the true value within its posterior')
axis.legend()
plt.show()

---

# 7. Prior against posterior

The same parameter drawn twice: from the prior, and from the posterior. The prior draws are the ones already generated in section 2, so this is the prior that was actually sampled, including the Dirichlet allocation and the derived pools.

Two numbers per parameter:

- **`sd ratio` = posterior sd / prior sd.** In words: how much narrower the data made the distribution. A ratio near 1 means the data did not narrow it.
- **`shift`** = (posterior mean - prior mean) / prior sd. In words: how far the centre moved, measured in prior standard deviations.

Neither number is a verdict. A parameter whose posterior sits exactly on its prior is the signature this project is looking for in RQ3, but it is also what you would see if the sampler never moved, which section 5 is what tells you about.

In [ ]:
# Parameters present in both the prior draws and the posterior.
COMPARABLE = [c for c in PRIOR_FRAME.columns
              if c != 'draw' and c in POST.columns]
print('comparable:', len(COMPARABLE))
print('prior only:', [c for c in PRIOR_FRAME.columns
                      if c not in POST.columns and c != 'draw'])
print('posterior only:', [c for c in POST_PARAMETERS
                          if c not in PRIOR_FRAME.columns])

rows = []
for name in COMPARABLE:
    prior_values = PRIOR_FRAME[name].to_numpy()
    post_values = POST[name].to_numpy()
    prior_sd = float(prior_values.std(ddof=1))
    rows.append({
        'parameter': name,
        'prior mean': float(prior_values.mean()),
        'prior sd': prior_sd,
        'post mean': float(post_values.mean()),
        'post sd': float(post_values.std(ddof=1)),
        'sd ratio': float(post_values.std(ddof=1) / prior_sd) if prior_sd > 0 else np.nan,
        'shift': float((post_values.mean() - prior_values.mean()) / prior_sd)
        if prior_sd > 0 else np.nan,
    })

MOVEMENT = pd.DataFrame(rows).set_index('parameter').sort_values('sd ratio')
MOVEMENT

In [ ]:
# Prior and posterior overlaid, one panel per parameter. Prior bounds drawn
# where the parameter has them.
N_COLS = 4
n_rows = int(np.ceil(len(COMPARABLE) / N_COLS))
figure, axes = plt.subplots(n_rows, N_COLS, figsize=(3.4 * N_COLS, 2.3 * n_rows))
axes = np.atleast_1d(axes).ravel()

for axis, name in zip(axes, COMPARABLE):
    prior_values = PRIOR_FRAME[name].to_numpy()
    post_values = POST[name].to_numpy()
    axis.hist(prior_values, bins=35, density=True, alpha=0.55,
              color=PLOT_COLOURS['prior'], label='prior')
    axis.hist(post_values, bins=35, density=True, alpha=0.65,
              color=PLOT_COLOURS['posterior'], label='posterior')
    if name in TRUE_VALUES:
        axis.axvline(TRUE_VALUES[name], color=PLOT_COLOURS['truth'], lw=1.6)
    if name in SAMPLED_BOUNDS:
        for edge in SAMPLED_BOUNDS[name]:
            axis.axvline(edge, color='0.35', lw=0.8, ls='--')
    axis.set_title(name + '  sd ratio '
                   + format(MOVEMENT.loc[name, 'sd ratio'], '.2f'))
    axis.set_yticks([])
for axis in axes[len(COMPARABLE):]:
    axis.set_visible(False)
axes[0].legend(fontsize=7)
figure.suptitle('Prior (blue) against posterior (vermillion); dashed lines are '
                'the prior bounds, green is the truth')
plt.show()

---

# 8. Pairwise correlation

The correlation between two parameters across posterior draws. Formula: the Pearson coefficient `r = cov(x, y) / (sd(x) sd(y))`, which runs from -1 to 1. In words: how much knowing one parameter tells you about the other, within the posterior.

**A high correlation here does not have to mean a trade-off between the parameters.** It can also come from a **mode split**: chains settling in separate regions of parameter space and never crossing between them. Pool those chains and every parameter that differs between the regions appears correlated with every other one, because each draw carries the signature of which region it came from -- not because the two parameters trade off against each other inside either region.

This is not hypothetical here. `reports/sampling/FINDINGS_sampling.md` and `scripts/22_calibration_figures.py` record the real-data calibration finding two modes about 3,955 log units apart with no draw bridging them, R-hat 1.50 and minimum ESS 7, while each chain was individually well mixed.

So the next cells compute the correlation twice: **pooled** across all chains, and **within each chain separately**. A pair whose pooled correlation is high while its within-chain correlation is low is the pattern a mode split produces. Whether that is what you are looking at is your call; section 5's per-chain log posterior is the other half of the evidence.

In [ ]:
# Correlation matrices: pooled, and the largest magnitude found within any
# single chain.
CORR_COLUMNS = [c for c in POST_PARAMETERS if POST[c].std(ddof=1) > 0]
dropped = [c for c in POST_PARAMETERS if c not in CORR_COLUMNS]
print('constant in the posterior, dropped:', dropped or 'none')

CORR_POOLED = POST[CORR_COLUMNS].corr()

per_chain = []
for chain in range(N_CHAINS):
    block = POST.loc[POST['chain'] == chain, CORR_COLUMNS]
    per_chain.append(block.corr())
CORR_BY_CHAIN = per_chain
CORR_WITHIN_MAX = pd.concat(per_chain).abs().groupby(level=0).max()
CORR_WITHIN_MAX = CORR_WITHIN_MAX.loc[CORR_POOLED.index, CORR_POOLED.columns]

print('matrix:', CORR_POOLED.shape)
print('chains compared separately:', N_CHAINS)

In [ ]:
# The strongest pairs, pooled, with their within-chain counterpart alongside.
# stack() keeps NaN in pandas 3, so the lower triangle is dropped explicitly.
upper = np.triu(np.ones(CORR_POOLED.shape, dtype=bool), k=1)
pairs = (CORR_POOLED.where(upper)
         .stack()
         .dropna()
         .rename('pooled r')
         .reset_index())
pairs.columns = ['a', 'b', 'pooled r']
pairs['max within-chain |r|'] = [
    float(CORR_WITHIN_MAX.loc[a, b]) for a, b in zip(pairs['a'], pairs['b'])
]
pairs['pooled |r|'] = pairs['pooled r'].abs()
pairs['pooled minus within'] = pairs['pooled |r|'] - pairs['max within-chain |r|']

TOP_PAIRS = (pairs.sort_values('pooled |r|', ascending=False)
             .head(N_TOP_PAIRS)
             .reset_index(drop=True))
print('pairs compared:', len(pairs))
TOP_PAIRS[['a', 'b', 'pooled r', 'max within-chain |r|', 'pooled minus within']]

In [ ]:
# The pooled correlation matrix as a heatmap.
figure, axis = plt.subplots(figsize=(0.32 * len(CORR_COLUMNS) + 3,
                                     0.32 * len(CORR_COLUMNS) + 2))
image = axis.imshow(CORR_POOLED.to_numpy(), cmap='RdBu_r', vmin=-1.0, vmax=1.0)
axis.set_xticks(range(len(CORR_COLUMNS)))
axis.set_xticklabels(CORR_COLUMNS, rotation=90, fontsize=7)
axis.set_yticks(range(len(CORR_COLUMNS)))
axis.set_yticklabels(CORR_COLUMNS, fontsize=7)
axis.grid(False)
axis.set_title('Posterior correlation, all chains pooled')
figure.colorbar(image, ax=axis, shrink=0.8, label='r')
plt.show()

In [ ]:
# The top pairs as scatter plots, coloured by chain. A mode split shows as
# separate clouds; a trade-off shows as one elongated cloud.
N_SCATTER = min(8, len(TOP_PAIRS))
N_COLS = 4
n_rows = int(np.ceil(N_SCATTER / N_COLS))
figure, axes = plt.subplots(n_rows, N_COLS, figsize=(3.3 * N_COLS, 3.0 * n_rows))
axes = np.atleast_1d(axes).ravel()

for axis, (_, row) in zip(axes, TOP_PAIRS.head(N_SCATTER).iterrows()):
    for chain in range(N_CHAINS):
        selection = POST['chain'] == chain
        axis.scatter(POST.loc[selection, row['a']], POST.loc[selection, row['b']],
                     s=4, alpha=0.4, color=OKABE_ITO[chain % len(OKABE_ITO)],
                     label='chain ' + str(chain))
    axis.set_xlabel(row['a'])
    axis.set_ylabel(row['b'])
    axis.set_title('r = ' + format(row['pooled r'], '.3f'), fontsize=9)
for axis in axes[N_SCATTER:]:
    axis.set_visible(False)
axes[0].legend(fontsize=7, markerscale=2)
plt.show()

---

# 9. Posterior predictive check

A **posterior predictive check** runs the forward model with parameter sets drawn from the posterior and compares the simulated NEE against the data that was fitted -- here, the synthetic NEE.

The trace does not carry daily NEE unless the model was built with `track_fluxes=True`, because that stores one full time series per draw. Two ways to get it, and the next cell takes whichever is available:

1. Load a cached `.npz` from `SYNTHETIC_POSTPRED_PATH`, assumed to hold a `nee` array of shape (draws, days) and a matching `chain` array. This mirrors `results/posterior_predictive_hemisurface.npz`, which the real-data figures use.
2. Recompute it by running the NumPy forward model on `N_POSTPRED_DRAWS` thinned posterior draws.

**Route 2 uses the NumPy model while the sampler used the PyTensor one.** The project asserts these agree to about 1e-14 in `tests/test_model.py`; if that test is not currently passing, the comparison below is between two models rather than one.

**Residuals** are `modelled - observed` on assimilable days. Also plotted is the standardised residual `z = (modelled - observed) / sigma`, with sigma the per-day standard deviation in `nee_unc` -- the same quantity the likelihood used.

In [ ]:
# Build the posterior predictive NEE: load the cache, or recompute.

def dalec_parameters_from_row(row):
    """One posterior draw as a DalecParameters.

    The allocation fractions are stored as deterministics and should already
    close to one. If they do not, they are renormalised through the simplex
    constructor -- which CHANGES the draw slightly, so the count of repairs is
    reported.
    """
    fields = {name: float(row[name]) for name in PARAMETER_NAMES}
    total = sum(fields[name] for name in ALLOCATION_PARAMETERS)
    if abs(total - 1.0) <= 1e-9:
        return DalecParameters(**fields), False
    weights = np.array([fields[name] for name in ALLOCATION_WEIGHT_ORDER], dtype=float)
    weights = weights / weights.sum()
    rest = {k: v for k, v in fields.items() if k not in ALLOCATION_PARAMETERS}
    return (DalecParameters.from_allocation_simplex(
        f_auto=fields['f_auto'], allocation_weights=weights, **rest), True)


if Path(SYNTHETIC_POSTPRED_PATH).exists():
    cache = np.load(SYNTHETIC_POSTPRED_PATH)
    PP_NEE = cache['nee']
    PP_CHAIN = cache['chain'] if 'chain' in cache else np.zeros(PP_NEE.shape[0], int)
    print('loaded cached posterior predictive:', PP_NEE.shape, 'from',
          SYNTHETIC_POSTPRED_PATH)
else:
    print('no cache at', SYNTHETIC_POSTPRED_PATH, '-- recomputing')
    step = max(1, len(POST) // N_POSTPRED_DRAWS)
    selected = POST.iloc[::step].head(N_POSTPRED_DRAWS)
    PP_NEE = np.full((len(selected), SYNTH.n_days), np.nan)
    PP_CHAIN = selected['chain'].to_numpy()
    repaired = 0
    started = time.perf_counter()
    for position, (_, row) in enumerate(selected.iterrows()):
        params, was_repaired = dalec_parameters_from_row(row)
        repaired += int(was_repaired)
        PP_NEE[position] = run_dalec2(params, SYNTH, gpp_fn=ACM).nee
        if (position + 1) % PROGRESS_EVERY == 0:
            print('  draw', position + 1, 'of', len(selected), ' elapsed',
                  round(time.perf_counter() - started, 1), 's', flush=True)
    print('recomputed', PP_NEE.shape, 'in', round(time.perf_counter() - started, 1), 's')
    print('draws whose allocation closure needed repair:', repaired)

print('draws per chain:', pd.Series(PP_CHAIN).value_counts().sort_index().to_dict())

In [ ]:
# Simulated NEE against the synthetic NEE, over the whole block.
PP_PERCENTILES = (2.5, 50.0, 97.5)
bands = np.percentile(PP_NEE, PP_PERCENTILES, axis=0)
observed_nee = np.where(SYNTH.nee_mask, SYNTH.nee_obs, np.nan)

figure, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                            gridspec_kw={'height_ratios': [3, 2]})

axes[0].fill_between(SYNTH_DATES, bands[0], bands[2], color=PLOT_COLOURS['posterior'],
                     alpha=0.35, label='posterior predictive 2.5-97.5 per cent')
axes[0].plot(SYNTH_DATES, bands[1], color=PLOT_COLOURS['posterior'], lw=1.2,
             label='posterior predictive median')
axes[0].plot(SYNTH_DATES, observed_nee, color=PLOT_COLOURS['observed'], lw=0.8,
             label='synthetic NEE (assimilable days)')
axes[0].axhline(0.0, color='0.4', lw=0.8)
axes[0].set_ylabel('NEE, g C m$^{-2}$ d$^{-1}$')
axes[0].legend(ncol=3)
axes[0].set_title('Posterior predictive against the synthetic data, '
                  + str(PP_NEE.shape[0]) + ' draws')

# Same thing collapsed onto the day of year, which shows the seasonal cycle.
frame = pd.DataFrame({'doy': SYNTH.doy,
                      'median': bands[1],
                      'observed': observed_nee})
seasonal = frame.groupby('doy').median()
axes[1].plot(seasonal.index, seasonal['median'], color=PLOT_COLOURS['posterior'],
             label='posterior predictive median')
axes[1].plot(seasonal.index, seasonal['observed'], color=PLOT_COLOURS['observed'],
             label='synthetic NEE')
axes[1].axhline(0.0, color='0.4', lw=0.8)
axes[1].set_xlabel('day of year')
axes[1].set_ylabel('NEE, g C m$^{-2}$ d$^{-1}$')
axes[1].legend()
plt.show()

In [ ]:
# Residuals over time, on assimilable days only.
mask = SYNTH.nee_mask
residual = bands[1] - SYNTH.nee_obs          # median simulated minus observed
residual = np.where(mask, residual, np.nan)
sigma = np.where(mask & np.isfinite(SYNTH.nee_unc) & (SYNTH.nee_unc > 0),
                 SYNTH.nee_unc, np.nan)
standardised = residual / sigma

figure, axes = plt.subplots(3, 1, figsize=(12, 9),
                            gridspec_kw={'height_ratios': [2, 2, 1.4]})

axes[0].plot(SYNTH_DATES, residual, lw=0.6, color=PLOT_COLOURS['posterior'])
axes[0].axhline(0.0, color='0.4', lw=0.8)
axes[0].set_ylabel('residual, g C m$^{-2}$ d$^{-1}$')
axes[0].set_title('Residual: posterior predictive median minus synthetic NEE')

axes[1].plot(SYNTH_DATES, standardised, lw=0.6, color=PLOT_COLOURS['prior'])
for level in (-2.0, 0.0, 2.0):
    axes[1].axhline(level, color='0.4', lw=0.8,
                    ls='--' if level != 0.0 else '-')
axes[1].set_ylabel('residual / sigma')
axes[1].set_title('Standardised by the per-day sigma the likelihood used')

finite = np.isfinite(standardised)
axes[2].hist(standardised[finite], bins=60, color=PLOT_COLOURS['prior'], alpha=0.8)
axes[2].set_xlabel('residual / sigma')
axes[2].set_ylabel('days')
plt.show()

print('assimilable days             ', int(mask.sum()))
print('days with a usable sigma     ', int(np.isfinite(sigma).sum()))
print('mean residual                ', float(np.nanmean(residual)))
print('mean absolute residual       ', float(np.nanmean(np.abs(residual))))
print('root mean square residual    ',
      float(np.sqrt(np.nanmean(residual ** 2))))
print('mean standardised residual   ', float(np.nanmean(standardised)))
print('sd of standardised residual  ', float(np.nanstd(standardised, ddof=1)))
print('days beyond 2 sigma          ',
      int((np.abs(standardised[finite]) > 2.0).sum()), 'of', int(finite.sum()))

In [ ]:
# Residual structure by season and by year, since a residual series that
# averages to zero can still be wrong in a pattern.
residual_frame = pd.DataFrame({
    'date': SYNTH_DATES,
    'year': SYNTH_DATES.year,
    'month': SYNTH_DATES.month,
    'residual': residual,
    'standardised': standardised,
}).dropna(subset=['residual'])

print('by month:')
print(residual_frame.groupby('month')[['residual', 'standardised']]
      .agg(['mean', 'std', 'count']).to_string())
print()
print('by year:')
print(residual_frame.groupby('year')[['residual', 'standardised']]
      .agg(['mean', 'std', 'count']).to_string())

---

# 10. Open questions this notebook can answer

The notebook produces the numbers. The answers are yours. Space is left under each.

### From the prior table (section 1)

**Q1. How many of the priors rest on a FI-Hyy measurement, how many on the published DALEC2 defaults, and how many on nothing yet?**

> 

**Q2. Which unverified sources have to be closed before the thesis quotes the prior table, and which can stay flagged?**

> 

**Q3. Does the `lma` prior change materially between the hemisurface and projected conventions, and does anything downstream depend on which is used?**

> 

### From the prior predictive check (sections 2 and 3)

**Q4. What fraction of prior draws produce a physically impossible forest, and which rule catches most of them?**

> 

**Q5. Which single parameter, if any, separates the failing draws from the passing ones?**

> 

**Q6. Does the simulated NEE fan contain the observed range, and where in the year does it not?**

> 

**Q7. Does the failure rate change when the draw count goes from 200 to 1000?**

> 

**Q8. Which of the branches in section 3 applies, and what is the next step?**

> 

### From the synthetic recovery (sections 4 to 7)

**Q9. Which parameters does the method recover -- truth inside the interval and the posterior visibly narrower than the prior?**

> 

**Q10. Which parameters have a posterior indistinguishable from their prior, and is that the same set the Morris screening flagged?**

> 

**Q11. Is any parameter recovered confidently but wrongly -- a narrow posterior with the truth outside it?**

> 

**Q12. Do the ranks cluster at 0 or 1 for any group of parameters, and do those parameters have anything in common?**

> 

### From the correlation and the predictive check (sections 8 and 9)

**Q13. Which parameter pairs are strongly correlated in the posterior?**

> 

**Q14. For each of those pairs, does the correlation survive within a single chain, or does it only appear when chains are pooled?**

> 

**Q15. Do the residuals have seasonal or interannual structure, and does it line up with the modelled growing season being about 41 days too long?**

> 

**Q16. Is the spread of the standardised residuals consistent with the noise that was actually added to the synthetic data?**

> 

### Crossing between sections

**Q17. Do the parameters the prior check identified as driving failures overlap with the ones the synthetic stage cannot recover?**

> 

**Q18. What would have to be true for this notebook's answer to RQ3 -- the limits of identifiability under NEE alone -- to be wrong?**

> 

---

# 11. Self-critique of this notebook's code

Written before you accept it. What is fragile, what was assumed, and what could give a wrong answer without saying so.

## Things that could be silently wrong

**1. The draws are not nested, and the failure rate will move when you change the count.**
`sample_reparameterised_parameters` draws each parameter as one vector of length `n_draws`. Raising `N_PRIOR_DRAWS` from 200 to 1000 therefore changes *every* draw, not just how many there are. A failure rate that shifts between the two is not evidence of anything; it is a different sample. Only a rate that is stable across several counts means something.

**2. The prior sampled here is the reparameterised one, and it is not the prior that produced the 83.7 per cent figure.**
The recorded 837-of-1000 failure came from the published Bloom and Williams priors. Comparing this notebook's rate against it is comparing two different priors. If you want the comparison, `dalec.diagnostics.sample_prior_parameters` draws the published set.

**3. `lai_above_ceiling` and `annual_nee_outside_range` are my thresholds, not measurements.**
The LAI ceiling is `SITE_LAI_MULTIPLE` (2.0, a round number I chose) times the site's measured seasonal maximum. The annual NEE window is a wide band I picked. Either can be tightened until any prior fails or loosened until none does. Both are in the config cell to make that visible rather than buried. The repository's own LAI band, used by the GPP magnitude gate, is the fixed `(1.0, 8.0)`.

**4. The annual NEE window compares against a site value the repository records twice, differently.**
`-215.8` in `scripts/22_calibration_figures.py` and about `-206` in a `parameters.py` comment, both attributed to Ilvesniemi et al. (2009). I did not resolve which is right; I set a window wide enough that the difference does not matter for the rule, which also means the rule is weak.

**5. The per-parameter failure breakdown is univariate and will miss joint causes.**
A failure driven by a *combination* of two parameters, each unremarkable alone, produces a low KS statistic on both. The heatmap will look empty and the cause will still be there. The reparameterisation of heterotrophic respiration exists precisely because `theta_som` and `c_som_0` were individually meaningless and jointly impossible -- exactly the pattern this breakdown cannot see.

**6. Slicing the block by year does not re-derive anything.**
`PRIOR_PRED_YEARS` cuts the driver record but the initial carbon pools each draw carries are still applied at day one of whatever is left. Starting in 1997 and starting in 2003 apply the same initial state to different weather. For a prior predictive check that is defensible; for anything comparing years against each other it is not.

**7. The `ran` and `evaluable` masks decide which draws each rule is applied to, and a mistake there moves every percentage.**
Draws that raised are counted once, under `integration_raised`, and excluded from the physical rules. If a draw runs but produces NaN partway through, it lands in `non_finite_flux` and is likewise excluded. That is a choice: the alternative, counting a NaN draw as failing every rule, would give higher and differently distributed percentages.

**8. `except Exception` in the prior run loop will swallow a genuine bug.**
It records the type and message in `prior_run['error']`, which is the mitigation, but a typo producing a `TypeError` on every draw would be reported as a 100 per cent failure rate rather than as a crash. If the failure rate is high, read that column before reading anything else.

**9. The ACM counters accumulate across draws.**
`ACM.clamp_count` and `ACM.range_floor_count` are attributes of one `AcmModel` instance shared by every draw, so the printed numbers are totals over the whole run, not per-draw diagnostics. Divide by `N_PRIOR_DRAWS` for a per-draw figure -- and note they carry over if you re-run the cell without rebuilding `ACM`.

**10. Section 9 may compare two different forward models.**
If the `.npz` cache is absent, the posterior predictive is recomputed with the NumPy model while the posterior came from the PyTensor one. The project asserts they agree to about 1e-14, but that assertion lives in `tests/test_model.py` and this notebook does not check it. Run `pytest tests/test_model.py` before trusting a residual structure you find here.

**11. The allocation closure repair in section 9 changes the draw.**
If a posterior draw's five allocation fractions do not sum to one within 1e-9, the code renormalises them and carries on. The count of repairs is printed. If it is not zero, something upstream is wrong and renormalising is hiding it rather than fixing it.

**12. The rank in section 6 is one rank from one synthetic dataset.**
It is not simulation-based calibration. A rank near 0 or 1 is consistent with a biased posterior, but it is also consistent with an unlucky draw of the noise. Distinguishing them needs many synthetic datasets, which is a different experiment.

**13. `truth in eti` uses an equal-tailed interval, not the highest-density interval.**
The config's `hdi_prob` names an HDI, and this notebook computes percentiles instead. For a skewed or multimodal posterior the two differ, and the ETI can exclude a truth the HDI would include. Change it if the project's reported intervals are HDIs.

**14. Section 8 compares pooled against within-chain correlation, which detects a mode split but does not prove one.**
A pair can show high pooled and low within-chain correlation for other reasons, including a slow-mixing chain that has not yet explored its own region. The per-chain log posterior in section 5 is the corroborating evidence, not this table alone.

## Assumptions that are written down but not verified

**15. The whole synthetic file contract is invented.**
`scripts/03_synthetic_twin.py` does not exist. Filenames, the JSON layout of the true parameters, the assumption that `SiteData.save` is the container, and the assumption that `nee_unc` carries the noise standard deviation actually used -- all are mine. Every one is marked `# TODO` in the config cell.

**16. The provenance categories in the prior table are my readings of `DECISIONS.md`.**
Three are arguable and are flagged in the markdown after the table: `ceff` filed as generic DALEC, `theta_lit` as boreal, `theta_som` as a site measurement when it is really a derivation from two site measurements.

**17. The unverified needle longevity is load-bearing and appears twice.**
The 3 to 5 year longevity has no citation in the repository, and both `c_lf` and `lma` are built from it. If it changes, the foliar pool and the canopy change together, which moves GPP.

## Things that are simply slow or awkward

**18. The forward run is a Python loop over days.**
Cost is linear in `N_PRIOR_DRAWS` times days. The default `PRIOR_PRED_YEARS = (1997, 1999)` keeps the first pass short; the full 1997-2010 block is about five times longer per draw.

**19. Memory.**
`prior_run['nee']` and `prior_run['gpp']` are each draws x days x 8 bytes. At 1000 draws over the full block that is about 41 MB each, which is fine; at 10,000 draws it is not. Full pool trajectories are deliberately not kept.

**20. `warnings.filterwarnings('ignore', category=RuntimeWarning)` in the config cell hides overflow and invalid-value warnings from numpy.**
That is convenient for a run where some prior draws legitimately blow up, and it will also hide a real numerical bug. Comment it out if a result looks strange.

## What this notebook does not do

It does not run the sampler, write any file, adjust any prior, or evaluate the GPP magnitude gate. It does not touch the real-data calibration except to offer its trace as a stand-in for exercising sections 5, 7 and 8, and that trace is recorded in the repository as not converged.